# Steam Game Network: Catalog Analysis

Exploratory analysis of 82,928 Steam games (2005–2025) from `steam_all_2005.json`.
Uses the packed array format: `[name, year, ratio, reviews, price, ratingIdx, genreIdxs, tagIdxs, developer]`

**Dataset**: [github.com/lukeslp/steam-network-data](https://github.com/lukeslp/steam-network-data)  
**Live viz**: [dr.eamer.dev/datavis/interactive/steam/](https://dr.eamer.dev/datavis/interactive/steam/)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
STEAM_BLUE = '#1b2838'
STEAM_LIGHT = '#66c0f4'
print('Libraries loaded')

## 1. Load Catalog

In [ ]:
with open('steam_all_2005.json') as f:
    catalog = json.load(f)

genres = catalog['genres']   # index -> genre name
tags = catalog['tags']       # index -> tag name
ratings = catalog['ratings'] # index -> rating label
games_raw = catalog['games']

# Unpack array format
# [name, year, ratio, reviews, price, ratingIdx, genreIdxs, tagIdxs, developer]
records = []
for g in games_raw:
    if len(g) < 9:
        continue
    records.append({
        'name': g[0],
        'year': g[1],
        'ratio': g[2],
        'reviews': g[3],
        'price': g[4],
        'rating': ratings[g[5]] if g[5] is not None and g[5] < len(ratings) else 'Unknown',
        'genres': [genres[i] for i in (g[6] or []) if i < len(genres)],
        'tags': [tags[i] for i in (g[7] or []) if i < len(tags)],
        'developer': g[8],
    })

df = pd.DataFrame(records)
print(f'Games loaded: {len(df):,}')
print(f'Years: {df.year.min()} - {df.year.max()}')
print(f'Genres: {len(genres)}, Tags: {len(tags)}, Ratings: {len(ratings)}')
df.head(3)

## 2. Release Year Trend (2005–2025)

In [ ]:
year_counts = df[df['year'].between(2005, 2025)]['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(year_counts.index, year_counts.values, color=STEAM_LIGHT, edgecolor=STEAM_BLUE, linewidth=0.5)
ax.set_title('Steam Games Released per Year (2005–2025)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Games Released')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(year_counts.index, rotation=45)
plt.tight_layout()
plt.show()
print(year_counts.to_string())

## 3. Rating Distribution

In [ ]:
rating_order = [
    'Overwhelmingly Positive', 'Very Positive', 'Positive', 'Mostly Positive',
    'Mixed', 'Mostly Negative', 'Negative', 'Very Negative', 'Overwhelmingly Negative', 'Unknown'
]
rating_counts = df['rating'].value_counts()
ordered = pd.Series({r: rating_counts.get(r, 0) for r in rating_order if r in rating_counts.index})

colors_map = {
    'Overwhelmingly Positive': '#1a9850', 'Very Positive': '#52ae32',
    'Positive': '#91cf60', 'Mostly Positive': '#d9ef8b',
    'Mixed': '#fee08b',
    'Mostly Negative': '#fc8d59', 'Negative': '#e34a33',
    'Very Negative': '#b30000', 'Overwhelmingly Negative': '#7f0000', 'Unknown': '#cccccc'
}
colors = [colors_map.get(r, '#999') for r in ordered.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(ordered.index, ordered.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Games by Review Rating', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Game Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, ordered.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Top Genres by Game Count and Review Volume

In [ ]:
genre_counts = Counter(g for genres_list in df['genres'] for g in genres_list)
genre_review_vol = {}
for _, row in df.iterrows():
    for g in row['genres']:
        genre_review_vol[g] = genre_review_vol.get(g, 0) + (row['reviews'] or 0)

top_n = 15
top_genres_count = pd.Series(genre_counts).nlargest(top_n).sort_values()
top_genres_vol = pd.Series(genre_review_vol).nlargest(top_n).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
top_genres_count.plot.barh(ax=ax1, color=STEAM_LIGHT, edgecolor=STEAM_BLUE, linewidth=0.5)
ax1.set_title('Top Genres by Game Count', fontsize=13, fontweight='bold')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax1.set_xlabel('Games')

top_genres_vol.plot.barh(ax=ax2, color='#FF6B35', edgecolor='white', linewidth=0.5)
ax2.set_title('Top Genres by Total Review Volume', fontsize=13, fontweight='bold')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax2.set_xlabel('Total Reviews')

plt.tight_layout()
plt.show()

## 5. Price Distribution

In [ ]:
df_priced = df.dropna(subset=['price'])
free = (df_priced['price'] == 0).sum()
paid = (df_priced['price'] > 0).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Free vs paid pie
ax1.pie([free, paid], labels=['Free', 'Paid'], autopct='%1.1f%%', colors=['#4CAF50', STEAM_LIGHT], startangle=90)
ax1.set_title(f'Free vs Paid (n={len(df_priced):,})', fontsize=13, fontweight='bold')

# Paid price histogram
paid_prices = df_priced[df_priced['price'] > 0]['price'].clip(0, 60)
ax2.hist(paid_prices, bins=40, color=STEAM_LIGHT, edgecolor=STEAM_BLUE, linewidth=0.3)
ax2.set_title('Paid Game Price Distribution', fontsize=13, fontweight='bold')
ax2.set_xlabel('Price (USD)')
ax2.set_ylabel('Count')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()
print(f'Free games: {free:,} ({free/len(df_priced)*100:.1f}%)')
print(f'Paid games: {paid:,} ({paid/len(df_priced)*100:.1f}%)')
print(f'Median paid price: ${paid_prices.median():.2f}')
print(f'Mean paid price: ${paid_prices.mean():.2f}')

## 6. Top 25 Games by Review Count

In [ ]:
top25 = df.nlargest(25, 'reviews')[['name', 'year', 'reviews', 'ratio', 'rating', 'price']].copy()
top25['reviews_M'] = (top25['reviews'] / 1e6).round(2)
top25['price_str'] = top25['price'].apply(lambda x: 'Free' if x == 0 else f'${x:.2f}' if pd.notna(x) else 'N/A')

fig, ax = plt.subplots(figsize=(14, 9))
colors = [('#1a9850' if 'Positive' in r else '#e34a33' if 'Negative' in r else '#fee08b') for r in top25['rating']]
bars = ax.barh(top25['name'][::-1], top25['reviews'][::-1], color=colors[::-1], edgecolor='white', linewidth=0.3)
ax.set_title('Top 25 Steam Games by Total Review Count', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Reviews')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

for bar, (_, row) in zip(bars[::-1], top25.iterrows()):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{row["ratio"]:.0f}% pos | {row["price_str"]}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(top25[['name','year','reviews_M','ratio','price_str']].to_string(index=False))

## 7. Genre Evolution Over Time (Stacked Area)

In [ ]:
top_genres = [g for g, _ in Counter(g for genres_list in df['genres'] for g in genres_list).most_common(8)]
df_year = df[df['year'].between(2005, 2025)].copy()

genre_year = pd.DataFrame(index=range(2005, 2026), columns=top_genres, data=0)
for _, row in df_year.iterrows():
    for g in row['genres']:
        if g in genre_year.columns:
            genre_year.loc[row['year'], g] += 1

fig, ax = plt.subplots(figsize=(14, 6))
genre_year.plot.area(ax=ax, alpha=0.8, cmap='tab10')
ax.set_title('Genre Composition by Year (Game Count)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Games Released')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

## 8. Tag Co-occurrence Heatmap (Top 20 Tags)

In [ ]:
tag_counts = Counter(t for tags_list in df['tags'] for t in tags_list)
top20_tags = [t for t, _ in tag_counts.most_common(20)]

cooccur = np.zeros((20, 20), dtype=int)
for tags_list in df['tags']:
    relevant = [t for t in tags_list if t in top20_tags]
    for i, t1 in enumerate(top20_tags):
        if t1 in relevant:
            for j, t2 in enumerate(top20_tags):
                if t2 in relevant:
                    cooccur[i, j] += 1

# Normalize diagonal to 1 for readability
diag = np.diag(cooccur)
cooccur_norm = cooccur / diag[:, None]
np.fill_diagonal(cooccur_norm, 1.0)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cooccur_norm, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.8)
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(top20_tags, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(top20_tags, fontsize=8)
ax.set_title('Tag Co-occurrence (normalized by row tag frequency)', fontsize=13, fontweight='bold', pad=12)
plt.colorbar(im, ax=ax, shrink=0.8, label='Co-occurrence rate')
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print('=== Steam Catalog Summary ===')
print(f'Total games: {len(df):,}')
print(f'Year range: {df.year.min()} - {df.year.max()}')
print(f'Unique developers: {df.developer.nunique():,}')
print(f'\nTop rating categories:')
for rating, count in df.rating.value_counts().head(6).items():
    print(f'  {rating}: {count:,} ({count/len(df)*100:.1f}%)')
print(f'\nReviews:')
print(f'  Total: {df.reviews.sum():,.0f}')
print(f'  Median per game: {df.reviews.median():,.0f}')
print(f'  Games with 0 reviews: {(df.reviews == 0).sum():,}')
print(f'  Games with 100K+ reviews: {(df.reviews >= 100000).sum():,}')